# Project: Customer Segmentation for E-Commerce

Segment customers using K-Means clustering and profile each segment.

In [ ]:
import sys, os, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_PATH = '/content/drive/MyDrive/data-science-mastery'
    if os.path.isdir(REPO_PATH):
        os.chdir(REPO_PATH)
        print(f'Working directory: {os.getcwd()}')
        if not os.path.isdir('Data') or len(os.listdir('Data')) < 5:
            subprocess.check_call([sys.executable, 'scripts/prepare_datasets.py'])
    else:
        REPO_PATH = '/content/data-science-mastery'
        if os.path.isdir(REPO_PATH):
            os.chdir(REPO_PATH)
        else:
            print('Repo not found. Run opencolab_setup.ipynb first.')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
print('Libraries loaded')

In [ ]:
np.random.seed(42)
n = 2000
df = pd.DataFrame({
    'annual_income': np.concatenate([
        np.random.normal(40, 8, 600), np.random.normal(70, 8, 700),
        np.random.normal(110, 12, 700)]).clip(15, 200),
    'spending_score': np.concatenate([
        np.random.normal(80, 8, 600), np.random.normal(45, 8, 700),
        np.random.normal(65, 10, 700)]).clip(0, 100),
    'purchase_frequency': np.concatenate([
        np.random.normal(15, 4, 600), np.random.normal(5, 2, 700),
        np.random.normal(10, 3, 700)]).clip(0, 30),
    'avg_order_value': np.concatenate([
        np.random.normal(30, 10, 600), np.random.normal(60, 15, 700),
        np.random.normal(45, 12, 700)]).clip(5, 200),
})
print('Dataset shape:', df.shape)
df.head()

## Find Optimal K

In [ ]:
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df)
inertias, silhouettes = [], []
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(df_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(df_scaled, labels))
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(range(2,11), inertias, 'o-', markersize=8)
axes[0].set_xlabel('K')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method')
axes[0].grid(True, alpha=0.3)
axes[1].plot(range(2,11), silhouettes, 'o-', markersize=8, color='green')
axes[1].set_xlabel('K')
axes[1].set_ylabel('Silhouette')
axes[1].set_title('Silhouette Analysis')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
best_k = range(2,11)[np.argmax(silhouettes)]
print(f'Optimal K (by silhouette): {best_k}')

## Apply K-Means and Visualize

In [ ]:
km = KMeans(n_clusters=3, random_state=42, n_init=10)
df['segment'] = km.fit_predict(df_scaled)
pca = PCA(n_components=2)
df[['pc1','pc2']] = pca.fit_transform(df_scaled)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
scatter = axes[0].scatter(df['pc1'], df['pc2'], c=df['segment'],
                          cmap='viridis', alpha=0.6, s=30)
axes[0].set_title('Customer Segments (PCA)')
plt.colorbar(scatter, ax=axes[0], label='Segment')
axes[0].grid(True, alpha=0.3)
for seg in range(3):
    sub = df[df['segment']==seg]
    axes[1].scatter(sub['annual_income'], sub['spending_score'],
                   label=f'Segment {seg}', alpha=0.5, s=20)
axes[1].set_xlabel('Annual Income')
axes[1].set_ylabel('Spending Score')
axes[1].set_title('Income vs Spending')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Profile Segments

In [ ]:
profile = df.groupby('segment')[['annual_income','spending_score',
                                  'purchase_frequency','avg_order_value']].mean()
print('Segment Profiles:')
print(profile.round(1))
for seg in range(3):
    sub = df[df['segment']==seg]
    print(f'\nSegment {seg} ({len(sub)} customers, {len(sub)/len(df)*100:.1f}%):')
    print(f'  Income: ${sub["annual_income"].mean():.0f}')
    print(f'  Spending: {sub["spending_score"].mean():.0f}/100')
    print(f'  Orders/mo: {sub["purchase_frequency"].mean():.1f}')

## Summary

- Generated customer data with 3 natural segments
- Used elbow + silhouette to find optimal K
- Applied K-Means, visualized with PCA
- Profiled each segment by behavior